# Sellers table: cleaning

Source : raw/olist_sellers_dataset.csv
Output : cleaned/sellers_clean.csv  (3,095 rows x 5 columns)

Decisions and findings for this table are written up in `cleaning_notes/README_sellers.md`.

### Setup

In [ ]:
import pandas as pd
from pathlib import Path

pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', None)
RAW = Path('raw')
CLEAN = Path('cleaned')
CLEAN.mkdir(exist_ok=True)

### 1. Load (zip prefix as text so leading zeros survive)

In [ ]:
sellers = pd.read_csv(RAW / 'olist_sellers_dataset.csv',
                      dtype={'seller_zip_code_prefix': str})
sellers['seller_zip_code_prefix'] = sellers['seller_zip_code_prefix'].str.zfill(5)

### 2. Checks (should show: unique ids, no nulls)

In [ ]:
print(sellers.shape)
print(sellers['seller_id'].is_unique)
print(sellers.isna().sum())
print(sellers['seller_city'].nunique())          # 611 before cleaning

### 3. Fix city text: odd apostrophe, accents, extra text after / \ , (

In [ ]:
c = sellers['seller_city'].str.replace('´', "'", regex=False)
c = c.str.normalize('NFKD').str.encode('ascii', errors='ignore').str.decode('ascii')
c = c.str.split(r'[/\\,(]', regex=True).str[0].str.strip()
sellers['city_was_fixed'] = c != sellers['seller_city']
sellers['seller_city'] = c

### 4. Fix values that are not real city names (run ONCE; use Restart + Run All to rerun)

In [ ]:
bad = ['sp', '04482255', 'vendas@creditparts.com.br']

# 4a. documented manual mapping: SBC = Sao Bernardo do Campo (zips 09726, 09861)
sbc_mask = sellers['seller_city'] == 'sbc'
sellers.loc[sbc_mask, 'seller_city'] = 'sao bernardo do campo'

# 4b. lookup: city used by other sellers with the same zip prefix
is_bad = sellers['seller_city'].isin(bad)
zip_city = (sellers.loc[~is_bad & ~sbc_mask]
            .groupby('seller_zip_code_prefix')['seller_city']
            .agg(lambda s: s.mode().iloc[0]))
sellers.loc[is_bad, 'seller_city'] = sellers.loc[is_bad, 'seller_zip_code_prefix'].map(zip_city)

# 4c. no match -> unknown
sellers['seller_city'] = sellers['seller_city'].fillna('unknown')

# 4d. flag every row touched
sellers['city_was_fixed'] = sellers['city_was_fixed'] | is_bad | sbc_mask
print(sellers['city_was_fixed'].sum())           # 29

### 5. Title-case cities (display only)

In [ ]:
sellers['seller_city'] = sellers['seller_city'].str.title()
print(sellers['seller_city'].nunique())          # 589

### 6. Link with sales (both should be 0)

In [ ]:
items = pd.read_csv(CLEAN / 'order_items_clean.csv', usecols=['order_id', 'seller_id'])
print((~items['seller_id'].isin(sellers['seller_id'])).sum())   # items from unknown sellers
print((~sellers['seller_id'].isin(items['seller_id'])).sum())   # sellers that never sold

### 7. Save and verify (keep this cell LAST)

In [ ]:
sellers.to_csv(CLEAN / 'sellers_clean.csv', index=False)

check = pd.read_csv(CLEAN / 'sellers_clean.csv', dtype={'seller_zip_code_prefix': str})
print(check.shape)                                            # (3095, 5)
print(check['seller_id'].is_unique)                           # True
print(check['seller_zip_code_prefix'].str.len().value_counts())  # all 5
print(check['seller_city'].nunique())                         # 589
print(check['city_was_fixed'].sum())                          # 29